In [30]:
# ============================================================
# 03_Model_Training.ipynb
# SIMPLE APPLE-LEVEL SPECTRAL TRAINING
# ============================================================

import os
import json
import numpy as np
import pandas as pd
from scipy.io import loadmat

from sklearn.model_selection import train_test_split
from sklearn.cluster import KMeans
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks

print("=" * 80)
print("SpectroFood Apple Maturity - SIMPLE Spectral-Only Training (Apple-level)")
print("=" * 80)
print(f"TensorFlow Version: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")

# ---------------- CONFIG ----------------
class Config:
    APPLE_MAT_PATH = "/content/drive/MyDrive/Colab Notebooks/dataset/Apple.mat"               # A — already uploaded
    CSV_PATH = "/content/drive/MyDrive/Colab Notebooks/dataset/SpectroFood_dataset.csv"       # A — already uploaded

    NUM_APPLES = 240
    NUM_BANDS = 141
    NUM_CLASSES = 3

    TEST_SIZE = 0.2
    RANDOM_STATE = 42

    BATCH_SIZE = 16
    EPOCHS = 200
    LEARNING_RATE = 1e-3

    CHECKPOINT_DIR = "checkpoints_simple"
    RESULTS_DIR = "results_simple"

config = Config()
os.makedirs(config.CHECKPOINT_DIR, exist_ok=True)
os.makedirs(config.RESULTS_DIR, exist_ok=True)


SpectroFood Apple Maturity - SIMPLE Spectral-Only Training (Apple-level)
TensorFlow Version: 2.19.0
GPU Available: []


In [31]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [32]:
# ============================================================
# OPTIONAL: GCS DOWNLOAD (YOU DO NOT NEED THIS NOW)
# ============================================================

"""
from google.colab import auth
from google.cloud import storage

auth.authenticate_user()
storage_client = storage.Client()
bucket = storage_client.bucket("processed_data-iyed")

def download_from_gcs(gcs_path, local_path):
    blob = bucket.blob(gcs_path)
    blob.download_to_filename(local_path)
    print(f"Downloaded {gcs_path} → {local_path}")

# Example:
# download_from_gcs("preprocessed/Apple.mat", "/content/Apple.mat")
# download_from_gcs("preprocessed/SpectroFood_dataset.csv", "/content/SpectroFood_dataset.csv")
"""


'\nfrom google.colab import auth\nfrom google.cloud import storage\n\nauth.authenticate_user()\nstorage_client = storage.Client()\nbucket = storage_client.bucket("processed_data-iyed")\n\ndef download_from_gcs(gcs_path, local_path):\n    blob = bucket.blob(gcs_path)\n    blob.download_to_filename(local_path)\n    print(f"Downloaded {gcs_path} → {local_path}")\n\n# Example:\n# download_from_gcs("preprocessed/Apple.mat", "/content/Apple.mat")\n# download_from_gcs("preprocessed/SpectroFood_dataset.csv", "/content/SpectroFood_dataset.csv")\n'

In [33]:
# ============================================================
# CELL 3: LOAD APPLE DATA
# ============================================================

print("\n[1/5] Loading Apple.mat and SpectroFood_dataset.csv ...")

# ---- Load Apple.mat ----
mat_data = loadmat(config.APPLE_MAT_PATH)

apples = {}
for i in range(1, config.NUM_APPLES + 1):
    key = f"A{i}"
    if key not in mat_data:
        raise KeyError(f"Missing {key} in Apple.mat")

    apples[i - 1] = mat_data[key].astype(np.float32)

print(f"✓ Loaded {len(apples)} apples from Apple.mat")

# ---- Load CSV ----
csv_data = pd.read_csv(config.CSV_PATH)

# Filter out rows where the 'Apple' column is NaN
# This addresses the AttributeError when .strip() is called on a float (NaN)
csv_data = csv_data.dropna(subset=['Apple'])

apple_to_dm = {}
for _, row in csv_data.iterrows():
    apple_name = row["Apple"].strip()  # "A1"
    apple_id = int(apple_name[1:]) - 1
    apple_to_dm[apple_id] = float(row["Dry matter"])

dm_values = np.array([apple_to_dm[i] for i in range(config.NUM_APPLES)], dtype=np.float32)
print(f"✓ Loaded dry matter values. Range = {dm_values.min():.2f} to {dm_values.max():.2f}")


[1/5] Loading Apple.mat and SpectroFood_dataset.csv ...
✓ Loaded 240 apples from Apple.mat
✓ Loaded dry matter values. Range = 0.13 to 0.17


In [34]:
# ============================================================
# CELL 4: APPLE-LEVEL SPECTRAL FEATURES
# ============================================================

print("\n[2/5] Computing mean spectral signature for each apple...")

spectral_features = []

for apple_id, cube in apples.items():
    H, W, B = cube.shape
    pixels = cube.reshape(-1, B)
    mean_vec = pixels.mean(axis=0)
    spectral_features.append(mean_vec)

X_all = np.stack(spectral_features, axis=0)
y_dm = dm_values

print(f"✓ X_all shape = {X_all.shape}  # (240, 141)")



[2/5] Computing mean spectral signature for each apple...
✓ X_all shape = (240, 141)  # (240, 141)


In [35]:
# ============================================================
# CELL 5: KMEANS → 3 MATURITY CLASSES
# ============================================================

print("\n[3/5] Running KMeans on dry matter values...")

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
clusters = kmeans.fit_predict(y_dm.reshape(-1, 1))

cluster_means = [y_dm[clusters == c].mean() for c in range(3)]
order = np.argsort(cluster_means)

# Map clusters → class labels
labels = np.zeros_like(clusters)
for new_label, old_cluster in enumerate(order):
    labels[clusters == old_cluster] = new_label

print("✓ Class mapping (sorted by ripeness):")
for new_label, old_cluster in enumerate(order):
    print(f"  Class {new_label} ← cluster {old_cluster}, mean DM = {cluster_means[old_cluster]:.3f}")

print("Class counts:", np.bincount(labels))



[3/5] Running KMeans on dry matter values...
✓ Class mapping (sorted by ripeness):
  Class 0 ← cluster 1, mean DM = 0.144
  Class 1 ← cluster 0, mean DM = 0.155
  Class 2 ← cluster 2, mean DM = 0.164
Class counts: [63 99 78]


In [36]:
# ============================================================
# CELL 6: SPLIT + NORMALIZE
# ============================================================

print("\n[4/5] Train/test split + scaling...")

idx = np.arange(config.NUM_APPLES)

X_train, X_test, y_train, y_test = train_test_split(
    X_all, labels,
    test_size=config.TEST_SIZE,
    stratify=labels,
    random_state=config.RANDOM_STATE
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

y_train_cat = keras.utils.to_categorical(y_train, 3)
y_test_cat = keras.utils.to_categorical(y_test, 3)

print("✓ X_train:", X_train_scaled.shape)
print("✓ X_test:", X_test_scaled.shape)

# --- Save scaler parameters for later use (POST‑TRAINING_DEMO) ---
np.save(os.path.join(config.RESULTS_DIR, "scaler_mean.npy"), scaler.mean_)
np.save(os.path.join(config.RESULTS_DIR, "scaler_scale.npy"), scaler.scale_)

print(f"✓ Saved scaler_mean.npy and scaler_scale.npy in {config.RESULTS_DIR}")



[4/5] Train/test split + scaling...
✓ X_train: (192, 141)
✓ X_test: (48, 141)
✓ Saved scaler_mean.npy and scaler_scale.npy in results_simple


In [37]:
# ============================================================
# CELL 7: TRAIN SIMPLE MLP
# ============================================================

print("\n[5/5] Training spectral-only MLP...")

model = keras.Sequential([
    layers.Input(shape=(config.NUM_BANDS,)),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(64, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(3, activation="softmax"),
])

model.compile(
    optimizer=keras.optimizers.Adam(config.LEARNING_RATE),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

checkpoint_path = os.path.join(config.CHECKPOINT_DIR, "best_model.h5")

cb = [
    callbacks.ModelCheckpoint(checkpoint_path, save_best_only=True, monitor="val_accuracy", mode="max"),
    callbacks.EarlyStopping(monitor="val_accuracy", patience=20, restore_best_weights=True),
    callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=10),
]

history = model.fit(
    X_train_scaled, y_train_cat,
    validation_data=(X_test_scaled, y_test_cat),
    epochs=config.EPOCHS,
    batch_size=config.BATCH_SIZE,
    callbacks=cb,
    verbose=1
)



[5/5] Training spectral-only MLP...
Epoch 1/200
 1/12 ━━━━━━━━━━━━━━━━━━━━ 11s 1s/step - accuracy: 0.3125 - loss: 1.4528

12/12 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - accuracy: 0.3610 - loss: 1.3454 - val_accuracy: 0.3333 - val_loss: 1.2798 - learning_rate: 0.0010
Epoch 2/200
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.3808 - loss: 1.2741 - val_accuracy: 0.3125 - val_loss: 1.1886 - learning_rate: 0.0010
Epoch 3/200
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.4139 - loss: 1.0946 - val_accuracy: 0.3333 - val_loss: 1.1455 - learning_rate: 0.0010
Epoch 4/200
 1/12 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.5000 - loss: 1.0418

12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.5161 - loss: 1.0458 - val_accuracy: 0.3542 - val_loss: 1.0957 - learning_rate: 0.0010
Epoch 5/200
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.4620 - loss: 1.1188 - val_accuracy: 0.2917 - val_loss: 1.1369 - learning_rate: 0.0010
Epoch 6/200
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.4946 - loss: 0.9845 - val_accuracy: 0.3333 - val_loss: 1.1113 - learning_rate: 0.0010
Epoch 7/200
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.4489 - loss: 1.0377 - val_accuracy: 0.2917 - val_loss: 1.1274 - learning_rate: 0.0010
Epoch 8/200
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.4811 - loss: 1.0404 - val_accuracy: 0.2500 - val_loss: 1.1140 - learning_rate: 0.0010
Epoch 9/200
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.4606 - loss: 1.0761 - val_accuracy: 0.2917 - val_loss: 1.1151 - learning_rate: 0.0010
Epoch 10/200
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.4453 - loss: 1.0157 - val_accuracy: 0.29

12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.4984 - loss: 0.9737 - val_accuracy: 0.3750 - val_loss: 1.0986 - learning_rate: 0.0010
Epoch 14/200
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.5397 - loss: 0.9529 - val_accuracy: 0.3333 - val_loss: 1.1353 - learning_rate: 0.0010
Epoch 15/200
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.4418 - loss: 1.1212 - val_accuracy: 0.3542 - val_loss: 1.1041 - learning_rate: 0.0010
Epoch 16/200
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.5001 - loss: 0.9922 - val_accuracy: 0.3333 - val_loss: 1.1325 - learning_rate: 0.0010
Epoch 17/200
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.5328 - loss: 0.9382 - val_accuracy: 0.3333 - val_loss: 1.1441 - learning_rate: 0.0010
Epoch 18/200
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.5196 - loss: 0.9885 - val_accuracy: 0.3333 - val_loss: 1.1217 - learning_rate: 0.0010
Epoch 19/200
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.4922 - loss: 1.0313 - val_accuracy

12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.5077 - loss: 0.9467 - val_accuracy: 0.3958 - val_loss: 1.1065 - learning_rate: 5.0000e-04
Epoch 25/200
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.5479 - loss: 0.9489 - val_accuracy: 0.3333 - val_loss: 1.0993 - learning_rate: 5.0000e-04
Epoch 26/200
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.5106 - loss: 0.9920 - val_accuracy: 0.3333 - val_loss: 1.1175 - learning_rate: 5.0000e-04
Epoch 27/200
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.5438 - loss: 0.9448 - val_accuracy: 0.3333 - val_loss: 1.1171 - learning_rate: 5.0000e-04
Epoch 28/200
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.5385 - loss: 0.9207 - val_accuracy: 0.3958 - val_loss: 1.0964 - learning_rate: 5.0000e-04
Epoch 29/200
 1/12 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - accuracy: 0.4375 - loss: 0.9681

12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.5114 - loss: 0.9464 - val_accuracy: 0.4167 - val_loss: 1.0991 - learning_rate: 5.0000e-04
Epoch 30/200
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.5504 - loss: 1.0185 - val_accuracy: 0.3750 - val_loss: 1.1052 - learning_rate: 5.0000e-04
Epoch 31/200
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.4953 - loss: 0.9818 - val_accuracy: 0.3750 - val_loss: 1.1142 - learning_rate: 5.0000e-04
Epoch 32/200
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.5531 - loss: 0.9273 - val_accuracy: 0.3542 - val_loss: 1.1164 - learning_rate: 2.5000e-04
Epoch 33/200
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.5044 - loss: 0.9531 - val_accuracy: 0.3750 - val_loss: 1.1185 - learning_rate: 2.5000e-04
Epoch 34/200
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.5210 - loss: 0.9043 - val_accuracy: 0.3750 - val_loss: 1.1259 - learning_rate: 2.5000e-04
Epoch 35/200
12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.5755 - loss

In [38]:
# ============================================================
# CELL 8: EVALUATION
# ============================================================

print("\n📊 Final Evaluation")

loss, acc = model.evaluate(X_test_scaled, y_test_cat)
print(f"Test accuracy = {acc:.4f}")

probs = model.predict(X_test_scaled)
preds = np.argmax(probs, axis=1)

print("\nClassification Report:")
print(classification_report(y_test, preds))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, preds))



📊 Final Evaluation
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.4549 - loss: 1.0605 


Test accuracy = 0.4167
1/2 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step

2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step

Classification Report:
              precision    recall  f1-score   support

           0       0.75      0.23      0.35        13
           1       0.44      0.55      0.49        20
           2       0.32      0.40      0.35        15

    accuracy                           0.42        48
   macro avg       0.50      0.39      0.40        48
weighted avg       0.49      0.42      0.41        48


Confusion Matrix:
[[ 3  5  5]
 [ 1 11  8]
 [ 0  9  6]]
